## Import of the required libraries


In [2]:
import json
import logging
import os

import numpy as np
import open3d as o3d
from SceneGraph3D import SceneGraph3D

%load_ext autoreload
%autoreload 2
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Constants definition


In [3]:
SSG_REPO = "3DSSG"  # Path to the 3DSSG repository
RSCAN_REPO = "3RScan"  # Path to the 3RScan repository
REL_FILE = os.path.join(SSG_REPO, "relationships.json")
OBJ_FILE = os.path.join(SSG_REPO, "objects.json")
SEMSEG_FILE = "semseg.v2.json"
PCD_FILE = "labels.instances.annotated.v2.ply"
MESH_FILE = "mesh.refined.v2.obj"
# change this to explore other scans (you can also cycle through them)
SCAN_ID = 882  # Example of one scan ID in 3RScan

## Load the 3DSSG dataset


In [4]:
scans = json.load(open(REL_FILE))["scans"]
objects = json.load(open(OBJ_FILE))["scans"]
objects = {obj["scan"]: obj for obj in objects}
for scan in scans:
    id_scan = scan["scan"]
    scan["objects"] = objects[id_scan]["objects"]
max_objects_num = 0
max_rel_num = 0
min_objects_num = 20
min_rel_num = 200
for scan in scans:
    if len(scan["relationships"]) == 0:
        print(scan)
    max_objects_num = max(max_objects_num, len(scan["objects"]))
    max_rel_num = max(max_rel_num, len(scan["relationships"]))
    min_objects_num = min(min_objects_num, len(scan["objects"]))
    min_rel_num = min(min_rel_num, len(scan["relationships"]))
print(max_objects_num, max_rel_num)  # 147 5167
print(min_objects_num, min_rel_num)  # 2 1

147 5167
2 1


In [5]:
print(scans[SCAN_ID]["objects"][0].keys())
print(scans[SCAN_ID]["relationships"])

dict_keys(['ply_color', 'nyu40', 'eigen13', 'label', 'rio27', 'affordances', 'id', 'global_id', 'attributes'])
[[2, 1, 15, 'standing on'], [5, 1, 14, 'attached to'], [6, 1, 14, 'attached to'], [7, 1, 15, 'standing on'], [8, 2, 15, 'standing on'], [9, 2, 1, 'supported by'], [10, 1, 15, 'standing on'], [11, 10, 16, 'lying on'], [2, 7, 6, 'close by'], [2, 10, 3, 'right'], [2, 10, 6, 'close by'], [7, 2, 6, 'close by'], [7, 10, 3, 'right'], [7, 10, 6, 'close by'], [8, 9, 5, 'behind'], [8, 9, 6, 'close by'], [8, 9, 2, 'left'], [9, 8, 4, 'front'], [9, 8, 3, 'right'], [9, 8, 6, 'close by'], [10, 2, 6, 'close by'], [10, 2, 2, 'left'], [10, 7, 6, 'close by'], [10, 7, 2, 'left'], [5, 6, 32, 'same object type'], [10, 7, 32, 'same object type'], [6, 5, 32, 'same object type'], [7, 10, 32, 'same object type']]


## Instantiating the Graph


In [6]:
g = SceneGraph3D.from_dict(scans[SCAN_ID])
scan_id = g.scan_id
file_3d_path = os.path.join(RSCAN_REPO, scan_id, SEMSEG_FILE)
file_pointcloud = os.path.join(RSCAN_REPO, scan_id, PCD_FILE)
file_mesh = os.path.join(RSCAN_REPO, scan_id, MESH_FILE)
g.read_3d_scene(file_3d_path, file_pointcloud, file_mesh)


### Printing all the nodes id of the graph


In [9]:
objects = g.nodes(data=True)
print("List of objects:\n", objects)

List of objects:
 [(1, {'ply_color': '#aec7e8', 'nyu40': '2', 'eigen13': '5', 'label': 'floor', 'rio27': '2', 'affordances': ['placing items on', 'walking on'], 'id': '1', 'global_id': '188', 'attributes': {'texture': ['tiled'], 'shape': ['flat'], 'lexical': ['inside', 'lower', 'horizontal'], 'state': ['clean', 'tidy']}}), (2, {'ply_color': '#1f77b4', 'nyu40': '7', 'eigen13': '10', 'label': 'table', 'rio27': '7', 'affordances': ['placing items on', 'cleaning', 'carrying'], 'id': '2', 'global_id': '455', 'attributes': {}}), (3, {'ply_color': '#ffbb78', 'nyu40': '29', 'eigen13': '7', 'label': 'boxes', 'rio27': '20', 'affordances': ['placing items in', 'placing items on', 'throwing away', 'moving'], 'id': '3', 'global_id': '60', 'attributes': {'lexical': ['rectangular']}}), (4, {'ply_color': '#ff7f0e', 'nyu40': '39', 'eigen13': '6', 'label': 'drawers rack', 'rio27': '0', 'affordances': ['placing items in'], 'id': '4', 'global_id': '163', 'attributes': {}}), (5, {'ply_color': '#98df8a', 'n

In [10]:
position_5 = g.get_node_centroid(5)
print(position_5)

[0.33765129534917765, 1.2809999881968785, -0.7297250672381751]


### Printing all the edges of the graph


In [11]:
edges = g.edges(data=True)
print("List of edges:\n", edges)
print("Num edges:", len(edges))

List of edges:
 [(2, 1, {'id': 15, 'name': 'standing on'}), (2, 7, {'id': 6, 'name': 'close by'}), (2, 10, {'id': 6, 'name': 'close by'}), (5, 1, {'id': 14, 'name': 'attached to'}), (5, 6, {'id': 32, 'name': 'same object type'}), (6, 1, {'id': 14, 'name': 'attached to'}), (6, 5, {'id': 32, 'name': 'same object type'}), (7, 1, {'id': 15, 'name': 'standing on'}), (7, 2, {'id': 6, 'name': 'close by'}), (7, 10, {'id': 32, 'name': 'same object type'}), (8, 2, {'id': 15, 'name': 'standing on'}), (8, 9, {'id': 2, 'name': 'left'}), (9, 2, {'id': 1, 'name': 'supported by'}), (9, 8, {'id': 6, 'name': 'close by'}), (10, 1, {'id': 15, 'name': 'standing on'}), (10, 2, {'id': 2, 'name': 'left'}), (10, 7, {'id': 32, 'name': 'same object type'}), (11, 10, {'id': 16, 'name': 'lying on'})]
Num edges: 18


### Printing the names of the entities of the graph


In [12]:
for obj in objects:
    print("e -> ", obj[1]["label"])

e ->  floor
e ->  table
e ->  boxes
e ->  drawers rack
e ->  wall
e ->  wall
e ->  chair
e ->  microwave
e ->  toaster
e ->  chair
e ->  item


### Printing the relations of the edge


In [13]:
for edj in edges:
    e1 = g[edj[0]]["label"]
    e2 = g[edj[1]]["label"]
    edj_name = edj[2]["name"]
    print(e1, " -> ", edj[2]["name"], " -> ", e2)
print("Num objects:", len(objects))

table  ->  standing on  ->  floor
table  ->  close by  ->  chair
table  ->  close by  ->  chair
wall  ->  attached to  ->  floor
wall  ->  same object type  ->  wall
wall  ->  attached to  ->  floor
wall  ->  same object type  ->  wall
chair  ->  standing on  ->  floor
chair  ->  close by  ->  table
chair  ->  same object type  ->  chair
microwave  ->  standing on  ->  table
microwave  ->  left  ->  toaster
toaster  ->  supported by  ->  table
toaster  ->  close by  ->  microwave
chair  ->  standing on  ->  floor
chair  ->  left  ->  table
chair  ->  same object type  ->  chair
item  ->  lying on  ->  chair
Num objects: 11


In [14]:
# I want to count the different types of edges
edge_types = {}
for edj in edges:
    e1 = g[edj[0]]["label"]
    e2 = g[edj[1]]["label"]
    edj_name = edj[2]["name"]
    if edj_name not in edge_types:
        edge_types[edj_name] = 1
    else:
        edge_types[edj_name] += 1
print("Edge types and counts:")
for k, v in edge_types.items():
    print(f"{k}: {v}")


Edge types and counts:
standing on: 4
close by: 4
attached to: 2
same object type: 4
left: 2
supported by: 1
lying on: 1


#### Plot the graph in 2D

You may need to install posix graphviz https://stackoverflow.com/questions/35064304/runtimeerror-make-sure-the-graphviz-executables-are-on-your-systems-path-aft


In [ ]:
SceneGraph3D.render(g, "example_plot")  # renders to example_plot.pdf

#### Saving the graph as JSON (open and explore the JSON)


In [60]:
SceneGraph3D.to_json(g, "example_output.json")

### Visualize 3D Object semantic segmented


In [ ]:
vertex = g.pointcloud["vertex"]
xyz = np.vstack((vertex["x"], vertex["y"], vertex["z"])).T
colors = np.vstack((vertex["red"], vertex["green"], vertex["blue"])).T
obj_pcd = o3d.geometry.PointCloud()
obj_pcd.points = o3d.utility.Vector3dVector(xyz)
obj_pcd.colors = o3d.utility.Vector3dVector(colors / 255.0)
o3d.visualization.draw_plotly([obj_pcd])

In [ ]:
obj7 = g.get_object_pointcloud(7)  # get point cloud of object with node ID 7
o3d.visualization.draw_plotly([obj7])
# you could extract from this point cloud feautures with for example PointNet

### Visualize Mesh


In [9]:
o3d.visualization.draw([{"geometry": g.mesh, "name": "3RScan Mesh"}], show_ui=True)

FEngine (64 bits) created at 0x3035d4000 (threading is enabled)
FEngine resolved backend: OpenGL


[error] GLFW error: Cocoa: Failed to find service port for display


#### Mesh of an object


In [ ]:
obj7m = g.get_object_mesh(7)  # get point cloud of object with node ID 7
o3d.visualization.draw_plotly([obj7m])

In [ ]:
mesh_file = os.path.join(RSCAN_REPO, scan_id, MESH_FILE)
mesh = o3d.io.read_triangle_mesh(mesh_file)
tmesh = o3d.t.geometry.TriangleMesh.from_legacy(mesh)
seg_file = os.path.join(RSCAN_REPO, scan_id, SEMSEG_FILE)
data = json.load(open(seg_file))
# Create visualizer
submesh = mesh.select_by_index(data["segGroups"][0]["segments"], triangles=True)
# o3d.visualization.draw([{"geometry": tmesh, "name": "3RScan Mesh"}], show_ui=True)


TypeError: select_by_index(): incompatible function arguments. The following argument types are supported:
    1. (self: open3d.cpu.pybind.geometry.TriangleMesh, indices: List[int], cleanup: bool = True) -> open3d.cpu.pybind.geometry.TriangleMesh

Invoked with: TriangleMesh with 9506 points and 12479 triangles, and textures of size (4096, 4096) , [1527, 213, 2442, 826, 6235, 106]; kwargs: triangles=True

In [ ]:
from plyfile import PlyData

meshina = PlyData.read(open(mesh_file))

PlyHeaderParseError: line 1: expected 'ply'

In [27]:
inst_ply = os.path.join(RSCAN_REPO, scan_id, "labels.instances.annotated.v2.ply")

inst_mesh = o3d.io.read_triangle_mesh(inst_ply)
# instance_ids = np.asarray(
#     inst_mesh.vertex_attributes["instance_id"]
# ).astype(np.int32)

# print("Unique instance IDs:", np.unique(instance_ids))
o3d.visualization.draw(inst_mesh)
